# RFP vs Proposal Coverage Mapper: Day 3 (requirement extraction)

**Before running:**
1. **Add Input**: attach ONLY the dataset `rfp-coverage-mapper-final`
2. **Settings > Internet**: On · **Settings > Accelerator**: None
3. **Add-ons > Secrets**: tick `GEMINI_API_KEY` (needed today: Gemini reads the RFP)

Then click **Run All** (about 1–4 minutes; most of it is the Gemini call in Step 6).
Every step ends with **✅ STEP X DONE** or **❌ STEP X FAILED: reason**. A failed step stops the run on purpose.

## Step 1: Find the project and copy it

In [ ]:
# ===== STEP 1: find the project in the dataset and copy it to the working folder =====
import os, sys, glob, shutil

class StepFailed(Exception):
    pass

def fail(step, reason):
    print(f'❌ STEP {step} FAILED: {reason}')
    raise StepFailed(reason)   # stops Run All on purpose

PROJECT = '/kaggle/working/rfp-coverage-mapper'
candidates = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/check_setup.py', recursive=True)]
complete = [c for c in candidates if os.path.isfile(os.path.join(c, 'test_data/inputs/proposal_qamar.pptx'))]
print('Folders with check_setup.py:', candidates or 'none')
if not complete:
    fail(1, 'no project folder with test_data/ found. Is the dataset "rfp-coverage-mapper-final" attached (Add Input)?')
SOURCE = sorted(complete, key=lambda c: ('final' not in c, len(c)))[0]
print('Project found at:', SOURCE)

shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(SOURCE, PROJECT)
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
os.makedirs('evaluation', exist_ok=True)
print('Working in:', os.getcwd())

needed = ['test_data/inputs/rfp_qamar.pdf', 'test_data/ground_truth/answer_key.json', 'src/llm.py']
missing = [f for f in needed if not os.path.isfile(f)]
if missing:
    fail(1, f'files missing from the copy: {missing}')
print(f"Cache entries carried over: {len(glob.glob('cache/*.json'))}")
print('✅ STEP 1 DONE: project copied to', PROJECT)

## Step 2: Install libraries

In [ ]:
# ===== STEP 2: install libraries and check they import =====
import subprocess, importlib
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
                        capture_output=True, text=True)
if result.returncode != 0:
    fail(2, 'pip install failed: ' + result.stderr.strip()[-300:])
missing = []
for module in ['pdfplumber', 'docx', 'pptx', 'google.genai', 'pandas']:
    try:
        importlib.import_module(module)
    except Exception as e:
        missing.append(f'{module} ({e})')
if missing:
    fail(2, f'libraries not importable: {missing}')
print('(pip warnings about dopamine-rl / gradio are harmless and hidden here)')
print('✅ STEP 2 DONE: libraries ready')

## Step 3: API key and models

In [ ]:
# ===== STEP 3: load the API key from Kaggle Secrets and choose the models =====
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('GEMINI_API_KEY')
except Exception as e:
    fail(3, f'could not read GEMINI_API_KEY ({str(e)[:120]}). Fix: Add-ons > Secrets > tick GEMINI_API_KEY, then Run All again')
if not key:
    fail(3, 'GEMINI_API_KEY is empty. Fix: Add-ons > Secrets > check the value')
os.environ['GEMINI_API_KEY'] = key
os.environ['GEMINI_MODEL'] = 'gemini-3.8-flash'                 # main model
os.environ['GEMINI_FALLBACK_MODELS'] = 'gemini-3.5-flash-lite'  # used automatically if the main one is busy
print(f'Key loaded from Kaggle Secrets ({len(key)} characters, not shown)')
print('Main model: gemini-3.8-flash | backup: gemini-3.5-flash-lite')
print('✅ STEP 3 DONE: key and models set')

## Step 4: Save the code files and check them
Four cells save files (`Writing ...`); the last one checks they load. `parsers.py` is unchanged from Day 2.

In [ ]:
%%writefile src/parsers.py
"""
parsers.py - STAGE 1 of the tool: turn the input files into clean, structured text.

    parse_rfp(path)  -> the RFP (PDF or Word) as a list of "blocks"
    parse_deck(path) -> the proposal deck (PPTX) as a list of slides

A "block" is one heading, paragraph, bullet or table, tagged with the RFP
section it belongs to. Keeping this structure (instead of one big string)
means later stages can say WHERE in the RFP a requirement came from.

No AI is used here: parsing is deterministic, fast, free and repeatable.
"""
import re
from pathlib import Path

import pdfplumber
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE

# ---------------------------------------------------------------------------
# Safety: the tool must never read the answer key (prevents data leakage)
# ---------------------------------------------------------------------------
BLOCKED_FOLDERS = {"ground_truth"}


def check_allowed(path):
    """Refuse files inside ground_truth/ and files that don't exist."""
    p = Path(path)
    if any(part in BLOCKED_FOLDERS for part in p.parts):
        raise PermissionError(f"Refusing to read '{p}': the tool must never read the answer key (ground_truth).")
    if not p.is_file():
        raise FileNotFoundError(f"File not found: '{p}'")
    return p


# ---------------------------------------------------------------------------
# Text cleaning
# ---------------------------------------------------------------------------
CID = re.compile(r"\(cid:\d+\)")          # PDF symbols pdfplumber can't decode, e.g. bullets -> "(cid:127)"
BULLET_START = re.compile(r"^\s*(\(cid:\d+\)|[•▪◦●\-–*])\s*")
NUMBERED_HEADING = re.compile(r"^\d+(\.\d+)*\.?\s+[A-Z]")   # e.g. "2. Scope of Work"


def clean(text, keep_lines=False):
    """Remove undecodable symbols, non-breaking spaces and repeated spaces.
    keep_lines=True keeps line breaks (used for table rows)."""
    text = CID.sub("", str(text)).replace("\u00a0", " ")
    if keep_lines:
        lines = (re.sub(r"[ \t]+", " ", ln).strip() for ln in text.splitlines())
        return "\n".join(ln for ln in lines if ln)
    return re.sub(r"\s+", " ", text).strip()


def _make_block(blocks, kind, text, section):
    text = clean(text, keep_lines=(kind == "table"))
    if text:
        blocks.append({"id": f"B{len(blocks) + 1:03d}", "type": kind, "section": section, "text": text})


def _finish(blocks, path, fmt):
    """Package blocks into one result, plus a readable full text for the LLM."""
    lines = []
    for b in blocks:
        prefix = {"heading": "## ", "bullet": "• "}.get(b["type"], "")
        lines.append(prefix + b["text"])
    full_text = "\n\n".join(lines)
    return {
        "source": Path(path).name,
        "format": fmt,
        "sections": [b["text"] for b in blocks if b["type"] == "heading"],
        "blocks": blocks,
        "full_text": full_text,
        "word_count": len(full_text.split()),
    }


# ---------------------------------------------------------------------------
# RFP: Word (.docx)
# ---------------------------------------------------------------------------
def _docx_items(doc):
    """Paragraphs AND tables, in the order they appear in the document."""
    for child in doc.element.body.iterchildren():
        tag = child.tag.split("}")[-1]
        if tag == "p":
            yield Paragraph(child, doc)
        elif tag == "tbl":
            yield Table(child, doc)


def parse_docx(path):
    doc = Document(check_allowed(path))
    blocks, section = [], "Front matter"
    for item in _docx_items(doc):
        if isinstance(item, Table):
            rows = [" | ".join(clean(c.text) for c in row.cells) for row in item.rows]
            _make_block(blocks, "table", "\n".join(rows), section)
            continue
        style = item.style.name if item.style is not None else ""
        has_numbering = item._p.pPr is not None and item._p.pPr.numPr is not None
        if style.startswith("Heading"):
            section = clean(item.text) or section
            _make_block(blocks, "heading", item.text, section)
        elif style == "Title":
            _make_block(blocks, "paragraph", item.text, section)
        elif "List" in style or has_numbering:
            _make_block(blocks, "bullet", item.text, section)
        else:
            _make_block(blocks, "paragraph", item.text, section)
    return _finish(blocks, path, "docx")


# ---------------------------------------------------------------------------
# RFP: PDF
# ---------------------------------------------------------------------------
def parse_pdf(path):
    """
    PDFs store text as LINES, not paragraphs, so we rebuild them:
      - a numbered short line ("2. Scope of Work") starts a new section
      - a line starting with a bullet symbol starts a new bullet
      - a short line ending in . : ! ? closes the current paragraph
      - otherwise the line continues the current paragraph
    """
    with pdfplumber.open(check_allowed(path)) as pdf:
        lines = [ln for page in pdf.pages for ln in (page.extract_text() or "").splitlines() if ln.strip()]
    if not lines:
        raise ValueError(f"No text found in '{path}' (is it a scanned PDF?)")

    full_width = max(len(ln) for ln in lines)
    blocks, section = [], "Front matter"
    current, kind = [], "paragraph"

    def flush():
        nonlocal current
        if current:
            _make_block(blocks, kind, " ".join(current), section)
            current = []

    for ln in lines:
        text = ln.strip()
        if NUMBERED_HEADING.match(text) and len(text) < 80 and not text.endswith("."):
            flush()
            section = clean(text)
            _make_block(blocks, "heading", text, section)
            kind = "paragraph"
            continue
        if BULLET_START.match(text):
            flush()
            kind = "bullet"
            text = BULLET_START.sub("", text)
        current.append(text)
        if text.endswith((".", ":", "!", "?")) and len(ln) < 0.85 * full_width:
            flush()
            kind = "paragraph"
    flush()
    return _finish(blocks, path, "pdf")


def parse_rfp(path):
    """Read an RFP from .pdf or .docx (chosen by file extension)."""
    check_allowed(path)          # safety guard ALWAYS runs first, whatever the file type
    suffix = Path(path).suffix.lower()
    if suffix == ".pdf":
        return parse_pdf(path)
    if suffix == ".docx":
        return parse_docx(path)
    raise ValueError(f"Unsupported RFP format '{suffix}'. Use .pdf or .docx (save old .doc files as .docx).")


# ---------------------------------------------------------------------------
# Proposal deck: PowerPoint (.pptx)
# ---------------------------------------------------------------------------
def _walk_shapes(shapes):
    """All shapes in reading order (top to bottom, left to right), opening groups."""
    for shape in sorted(shapes, key=lambda s: ((s.top or 0), (s.left or 0))):
        if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            yield from _walk_shapes(shape.shapes)
        else:
            yield shape


def parse_deck(path):
    """
    One entry per slide: title, body lines, tables and speaker notes.
    'visible_text' = what the audience sees (title + body + tables).
    Speaker notes are kept separately: the client never sees them, so later
    stages can decide whether they count as "addressing" a requirement.
    """
    check_allowed(path)          # safety guard ALWAYS runs first
    if Path(path).suffix.lower() != ".pptx":
        raise ValueError(f"Unsupported deck format '{Path(path).suffix}'. Use .pptx (save old .ppt files as .pptx).")
    prs = Presentation(path)
    slides = []
    for number, slide in enumerate(prs.slides, start=1):
        title_shape = slide.shapes.title
        title = clean(title_shape.text_frame.text) if title_shape is not None and title_shape.has_text_frame else ""
        body, tables = [], []
        for shape in _walk_shapes(slide.shapes):
            if title_shape is not None and shape.shape_id == title_shape.shape_id:
                continue
            if shape.has_table:
                tables.append([[clean(c.text) for c in row.cells] for row in shape.table.rows])
            elif shape.has_text_frame:
                body.extend(t for t in (clean(p.text) for p in shape.text_frame.paragraphs) if t)
        notes = ""
        if slide.has_notes_slide:
            notes = clean(slide.notes_slide.notes_text_frame.text)

        parts = [f"Title: {title}"] + [f"- {b}" for b in body]
        for table in tables:
            parts += ["Table:"] + [" | ".join(row) for row in table]
        slides.append({
            "slide_number": number,
            "title": title,
            "body": body,
            "tables": tables,
            "notes": notes,
            "visible_text": "\n".join(parts),
        })
    return {"source": Path(path).name, "slide_count": len(slides), "slides": slides}


In [ ]:
%%writefile src/extract.py
"""
extract.py - STAGE 2 of the tool: find every requirement in the RFP.

Principle: "AI proposes, Python verifies."
  1. Gemini reads the RFP (as numbered blocks from parsers.py) and returns each
     requirement with a category, a short restatement, the EXACT quote and the
     block it came from.
  2. Python checks Gemini's work:
       - the quote must really exist in the RFP (catches invented text)
       - the RFP section is taken from the parser, not from Gemini
       - near-duplicate requirements are removed
       - a requirement whose quote is NOT in the RFP is set aside as "unverified"
         (saved and shown for human review, but not passed on), so the tool never
         reports a gap for something the client did not actually ask for
"""
import csv
import difflib
import json
import re
from pathlib import Path

from src.llm import MODEL, ask_json

CATEGORIES = ["Scope", "Deliverable", "Timeline", "Proposal requirement"]

SYSTEM = (
    "You are an experienced bid manager at a management consultancy. "
    "You read client RFPs and list precisely what the client requires, without adding anything."
)

INSTRUCTIONS = """Below is an RFP, split into numbered blocks. Each block shows its ID, type and RFP section.

TASK: list every requirement the client places on the consultant or on the proposal.

CATEGORIES (use exactly one):
- "Scope": work or activities the consultant must carry out
- "Deliverable": tangible outputs the consultant must hand over (reports, portfolios, presentations, language/format of outputs)
- "Timeline": durations, deadlines and milestones of the engagement itself
- "Proposal requirement": what the proposal document must contain or how it must be structured (e.g. CVs, pricing format)

RULES:
1. Keep a single ask together, even if it has several aspects (e.g. "assess the data infrastructure and its governance" is ONE requirement; later stages check each aspect).
2. Split lists of separate items into separate requirements (e.g. "a report, alongside a portfolio" = TWO deliverables).
3. List each requirement ONCE. If an ask is repeated elsewhere (e.g. in evaluation criteria or a summary), cite the place where it is stated as an obligation.
4. Do NOT extract: background or context, evaluation criteria and their weights, submission logistics (how, where or when to submit; clarification questions), confidentiality or legal terms.
5. "source_quote": copy the exact words from ONE block, verbatim, at most about 40 words. Never paraphrase the quote.
6. "requirement": a short, clear restatement (at most 25 words) starting with a verb.

Return JSON only:
{"requirements": [{"category": "...", "requirement": "...", "source_quote": "...", "block_id": "B000"}],
 "skipped_sections": [{"section": "...", "reason": "..."}]}

RFP BLOCKS:
"""


# ---------------------------------------------------------------------------
# Prompt
# ---------------------------------------------------------------------------
def build_prompt(rfp):
    """Show Gemini every block with its ID, type and section, so it can cite them."""
    lines = [f"[{b['id']} | {b['type']} | {b['section']}] {b['text']}" for b in rfp["blocks"]]
    return INSTRUCTIONS + "\n".join(lines)


# ---------------------------------------------------------------------------
# Python checks on Gemini's answer
# ---------------------------------------------------------------------------
def _norm(text):
    """Lower case, punctuation removed, single spaces: for fair text comparison."""
    text = re.sub(r"[^\w\s%]", " ", str(text).lower())
    return re.sub(r"\s+", " ", text).strip()


def quote_found(quote, text, threshold=0.9):
    """True if (almost) all of the quote appears, in order, inside the text."""
    q, t = _norm(quote), _norm(text)
    if not q:
        return False
    if q in t:
        return True
    matcher = difflib.SequenceMatcher(None, q, t, autojunk=False)
    matched = sum(block.size for block in matcher.get_matching_blocks())
    return matched / len(q) >= threshold


def _category(value):
    """Map Gemini's category onto our four fixed categories."""
    v = str(value).lower()
    for name, key in [("Deliverable", "deliver"), ("Timeline", "time"),
                      ("Proposal requirement", "proposal"), ("Scope", "scope")]:
        if key in v:
            return name
    return "Other"


def _is_duplicate(a, b):
    """Two requirements are duplicates if their statements or quotes are nearly the same."""
    same_text = difflib.SequenceMatcher(None, _norm(a["requirement"]), _norm(b["requirement"])).ratio() >= 0.85
    qa, qb = _norm(a["source_quote"]), _norm(b["source_quote"])
    same_quote = bool(qa and qb) and (qa in qb or qb in qa)
    return same_text or same_quote


def verify(raw_items, rfp):
    """Check each requirement against the real RFP text; return (kept, removed_duplicates, unverified)."""
    blocks = {b["id"]: b for b in rfp["blocks"]}
    order = {b["id"]: i for i, b in enumerate(rfp["blocks"])}
    checked = []
    for item in raw_items:
        if not isinstance(item, dict):
            continue
        quote = str(item.get("source_quote") or "").strip()
        match = re.search(r"B\d+", str(item.get("block_id") or ""))
        block_id = match.group(0) if match else ""

        if block_id in blocks and quote_found(quote, blocks[block_id]["text"]):
            status = "verified"
        else:
            found = next((b["id"] for b in rfp["blocks"] if quote_found(quote, b["text"])), None)
            if found:
                block_id, status = found, "verified (block corrected)"
            else:
                status = "NOT FOUND in RFP - review"

        checked.append({
            "category": _category(item.get("category")),
            "requirement": str(item.get("requirement") or "").strip(),
            "source_quote": quote,
            "block_id": block_id if block_id in blocks else "",
            "section": blocks[block_id]["section"] if block_id in blocks else "",
            "quote_check": status,
        })

    # set aside requirements whose quote is not in the RFP
    unverified = [r for r in checked if r["quote_check"].startswith("NOT FOUND")]
    checked = [r for r in checked if not r["quote_check"].startswith("NOT FOUND")]

    # keep RFP order, then remove near-duplicates (first one wins)
    checked.sort(key=lambda r: order.get(r["block_id"], 10**6))
    kept, removed = [], []
    for req in checked:
        twin = next((k for k in kept if _is_duplicate(k, req)), None)
        if twin:
            removed.append({**req, "duplicate_of": twin["requirement"]})
        else:
            kept.append(req)
    for i, req in enumerate(kept, start=1):
        req["id"] = f"R{i:02d}"
    return kept, removed, unverified


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------
def extract_requirements(rfp):
    """Parsed RFP in, verified requirement list out."""
    reply = ask_json(build_prompt(rfp), system=SYSTEM, temperature=0.0, cache_tag="extract-v1")
    if isinstance(reply, list):
        reply = {"requirements": reply}
    raw = reply.get("requirements") if isinstance(reply, dict) else None
    if not isinstance(raw, list) or not raw:
        raise ValueError("Gemini returned no requirements.")
    kept, removed, unverified = verify(raw, rfp)
    if not kept:
        raise ValueError("None of Gemini's requirements could be verified against the RFP text.")
    return {
        "rfp_source": rfp["source"],
        "model": MODEL,
        "requirements": kept,
        "removed_duplicates": removed,
        "unverified": unverified,
        "skipped_sections": reply.get("skipped_sections") or [],
    }


def save_requirements(result, folder="output"):
    """Save the full result as JSON, and the requirement list as CSV."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "requirements.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    fields = ["id", "category", "requirement", "section", "block_id", "source_quote", "quote_check"]
    with open(folder / "requirements.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(result["requirements"])
    return folder / "requirements.json", folder / "requirements.csv"


In [ ]:
%%writefile evaluation/__init__.py
# Evaluation code: the ONLY part of the project allowed to read test_data/ground_truth/


In [ ]:
%%writefile evaluation/check_extraction.py
"""
check_extraction.py - EVALUATION of stage 2 (requirement extraction).

This is NOT part of the tool. It is the "exam marking" step and the only kind
of code allowed to read the answer key in test_data/ground_truth/.

Question it answers: of the 18 requirements we planted in the RFP,
how many did the tool find (recall), and what else did it list?

Matching method (simple and transparent): for each planted requirement, find
the extracted requirement that shares the most content words with it.
score = shared content words / content words in the planted requirement.
A score of 0.5 or more counts as "found". Every match is printed, so a human
can check the matching itself.
"""
import json
import re
from pathlib import Path

STOPWORDS = set("""a an the and or of to for in on at by with from as is are be been all each
its it this that these those any into than then their our your shall must will should can
including include includes both least within over per""".split())


def content_words(text):
    """Meaningful words only (numbers are kept, e.g. '16', '500')."""
    return {w for w in re.findall(r"[a-z0-9]+", str(text).lower())
            if w.isdigit() or (len(w) > 2 and w not in STOPWORDS)}


def overlap(planted_text, extracted):
    planted = content_words(planted_text)
    found = content_words(extracted["source_quote"] + " " + extracted["requirement"])
    return len(planted & found) / len(planted) if planted else 0.0


def evaluate_extraction(result, answer_key_path="test_data/ground_truth/answer_key.json", threshold=0.5):
    planted = json.loads(Path(answer_key_path).read_text(encoding="utf-8"))
    extracted = result["requirements"]
    rows, used = [], {}
    for gt in planted:
        scored = sorted(((overlap(gt["text"], r), r) for r in extracted), key=lambda x: -x[0])
        score, best = scored[0] if scored else (0.0, None)
        hit = best is not None and score >= threshold
        if hit:
            used.setdefault(best["id"], []).append(gt["id"])
        rows.append({"planted_id": gt["id"], "category": gt["category"], "planted": gt["text"],
                     "found": hit, "match_id": best["id"] if hit else "", "score": round(score, 2),
                     "category_ok": hit and best["category"] == gt["category"]})
    merged = {rid: ids for rid, ids in used.items() if len(ids) > 1}
    extras = [r for r in extracted if r["id"] not in used]
    found = sum(r["found"] for r in rows)
    return {"rows": rows, "found": found, "total": len(planted), "recall": found / len(planted),
            "category_agreement": sum(r["category_ok"] for r in rows) / max(found, 1),
            "merged": merged, "extras": extras}


In [ ]:
# ===== STEP 4: check the code files were saved and load correctly =====
import importlib
try:
    import src.llm, src.parsers, src.extract, evaluation.check_extraction
    for module in (src.llm, src.parsers, src.extract, evaluation.check_extraction):
        importlib.reload(module)            # always use the versions just written
    from src.parsers import parse_rfp
    from src.extract import extract_requirements, save_requirements
    from evaluation.check_extraction import evaluate_extraction
except Exception as e:
    fail(4, f'code could not be loaded: {e}')
if src.llm.MODEL != 'gemini-3.8-flash':
    fail(4, f'unexpected model {src.llm.MODEL}; run Step 3 before Step 4')
print('Loaded: src/parsers.py, src/extract.py, evaluation/check_extraction.py | model:', src.llm.MODEL)
print('✅ STEP 4 DONE: code saved and loaded')

## Step 5: Parse the RFP

In [ ]:
# ===== STEP 5: parse the RFP (stage 1, from Day 2) =====
try:
    rfp = parse_rfp('test_data/inputs/rfp_qamar.pdf')
except Exception as e:
    fail(5, f'could not parse the RFP: {e}')
print(f"{rfp['source']}: {rfp['word_count']} words, {len(rfp['blocks'])} blocks, {len(rfp['sections'])} sections")
if len(rfp['blocks']) < 10:
    fail(5, 'too few blocks parsed - something is wrong with the RFP file')
print('✅ STEP 5 DONE: RFP parsed')

## Step 6: Extract requirements (Gemini + Python verification)

In [ ]:
# ===== STEP 6: extract requirements with Gemini, then verify them with Python (stage 2) =====
import pandas as pd
pd.set_option('display.max_colwidth', 90)
print('Asking Gemini to list the requirements (busy messages are normal; the backup model takes over if needed)...')
try:
    result = extract_requirements(rfp)
except Exception as e:
    fail(6, f'extraction failed: {str(e)[:300]}')
json_path, csv_path = save_requirements(result, 'output')
reqs = result['requirements']

table = pd.DataFrame(reqs)[['id', 'category', 'requirement', 'section', 'quote_check']]
display(table)
counts = table['category'].value_counts().to_dict()
print('Per category:', counts)
print(f"Removed as duplicates: {len(result['removed_duplicates'])} | set aside as unverified: {len(result['unverified'])}")
print('Sections Gemini skipped:', '; '.join(f"{s.get('section')} ({s.get('reason')})" for s in result['skipped_sections']) or 'none reported')
print(f'Saved: {json_path} and {csv_path}')
print(f'✅ STEP 6 DONE: {len(reqs)} requirements extracted and verified')

## Step 7: Quality checks (no answer key)

In [ ]:
# ===== STEP 7: quality checks (the tool's own checks; no answer key used) =====
warnings = []
def check(ok, label, warn_only=True):
    print(('  ✅ ' if ok else ('  ⚠️ ' if warn_only else '  ❌ ')) + label)
    if not ok:
        warnings.append(label)

check(len(reqs) >= 10, f'{len(reqs)} requirements found (expected at least 10 for this RFP)')
missing_cats = [c for c in ['Scope', 'Deliverable', 'Timeline', 'Proposal requirement'] if c not in counts]
check(not missing_cats, 'all 4 categories present' + (f' (missing: {missing_cats})' if missing_cats else ''))
exact = sum(r['quote_check'] == 'verified' for r in reqs)
corrected = len(reqs) - exact
check(True, f'every kept requirement quotes real RFP text ({exact} exact block, {corrected} block corrected by Python)')
check(not result['unverified'], f"no invented quotes ({len(result['unverified'])} set aside for review)"
      + ''.join(f"\n       - {u['requirement']}" for u in result['unverified']))
noise = [r for r in reqs if any(k in r['section'].lower() for k in ('evaluation', 'confidential', 'instructions', 'important dates'))]
check(not noise, 'nothing taken from evaluation criteria, instructions or confidentiality sections'
      + ''.join(f"\n       - {r['id']} from '{r['section']}'" for r in noise))
print(f"  ℹ️ duplicates removed by Python: {len(result['removed_duplicates'])}"
      + ''.join(f"\n       - '{d['requirement']}' (same as '{d['duplicate_of']}')" for d in result['removed_duplicates']))

if not reqs:
    fail(7, 'no requirements to check')
print(f'✅ STEP 7 DONE: quality checks complete ({len(warnings)} warning(s))')

## Step 8: Evaluation against the answer key

In [ ]:
# ===== STEP 8: EVALUATION against the answer key (the only step that reads ground_truth/) =====
ev = evaluate_extraction(result)
rows = pd.DataFrame(ev['rows'])[['planted_id', 'category', 'planted', 'found', 'match_id', 'score', 'category_ok']]
display(rows)
print(f"Recall: {ev['found']}/{ev['total']} planted requirements found ({ev['recall']:.0%})")
print(f"Category agreement on found requirements: {ev['category_agreement']:.0%}")
if ev['merged']:
    print('⚠️ One extracted requirement matched several planted ones (merged):', ev['merged'])
print(f"Extra requirements (in the RFP but not in our answer key; check by hand whether they are real asks): {len(ev['extras'])}")
for e in ev['extras']:
    print(f"   - {e['id']} [{e['category']}] {e['requirement']}  <- \"{e['source_quote'][:80]}\"")
not_found = [r['planted_id'] for r in ev['rows'] if not r['found']]
if not_found:
    print('⚠️ Planted requirements NOT found:', not_found)
print(f"✅ STEP 8 DONE: recall {ev['found']}/{ev['total']}")

## Step 9: Download

In [ ]:
# ===== STEP 9: save everything as a zip to download =====
zip_path = shutil.make_archive('/kaggle/working/rfp_coverage_mapper_day3', 'zip', '/kaggle/working', 'rfp-coverage-mapper')
size_kb = os.path.getsize(zip_path) // 1024
if size_kb == 0:
    fail(9, 'zip file is empty')
print(f'Saved {zip_path} ({size_kb} KB): code + test data + cache + output/')
print('✅ STEP 9 DONE: download rfp_coverage_mapper_day3.zip from the Output panel')